# Drug-combination interaction models (the interaction model as a model-selection axis)

**NOT FOR CLINICAL USE.** Population/regimen-level forward simulation only; no individual prognosis, no therapy ranking, no recommendation. Executed in CI (nbmake) so the result cannot silently break.

Oncology is overwhelmingly *combination* therapy, but a composed survival forecast for a combination silently depends on one unmeasured choice: how do the two drugs' effects combine? This notebook makes that choice a quantified model-selection axis — HSA vs additive (the Bliss/Loewe null) vs synergy — and shows how much the predicted outcome depends on it. Synergy is treated as a *declared assumption*, never estimated from data. See `onkos.interaction` and `docs/specs/research/combination-interaction.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos
from onkos.interaction import combine_effects, bliss_fraction, compare_interactions

ds = onkos.load()
print(f'{len(ds)} records | onkos {onkos.__version__}')

## The interaction rules

Three declared nulls combine two single-agent effect magnitudes `E_A`, `E_B`:

- `hsa` — highest single agent, `max(E_A, E_B)` (the conservative null);
- `additive` — Bliss-independence / effect-additive null, `E_A + E_B`;
- `greco` — interaction index, `E_A + E_B + ψ·√(E_A·E_B)` (ψ>0 synergy, ψ<0 antagonism, ψ=0 additive).

In [ ]:
ea, eb = 0.6, 0.6
for model, kw in [('hsa', {}), ('additive', {}), ('greco', dict(psi=0.5)), ('greco', dict(psi=-0.5))]:
    tag = model + (f"(psi={kw['psi']:+g})" if kw else '')
    print(f'{tag:<16} E_AB = {combine_effects(ea, eb, model=model, **kw):.3f}')

# Bliss independence equals additive kill rates for a log-linear process:
print('\nBliss == additive identity:', np.isclose(bliss_fraction(ea, eb), 1 - np.exp(-(ea + eb))))

## The divergence: the SAME single-agent activity, different predicted survival

For one combination (fixed `E_A`, `E_B`), simulate the population OS under every interaction model and measure the spread. This is the combination-therapy analog of the virtual-trial divergence view — the survival prediction depends on an interaction assumption you have not measured.

In [ ]:
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
t = np.linspace(0, 156, 313)
cmp = compare_interactions(ds, 'resistance.claret_2009.tgi', context=ctx,
                           effect_a=ea, effect_b=eb, psi=0.5, t=t)

print(f"tier {cmp.tier}  |  OS divergence across interaction models = {cmp.os_divergence:.3f}")
print(f"{'interaction':<14}{'combined E':>11}{'median OS':>11}")
for label, tr in cmp.trajectories.items():
    mos = f'{tr.median_os:.1f}' if tr.median_os else 'n/r'
    print(f'{label:<14}{cmp.combined_effects[label]:>11.3f}{mos:>11}')
lo, hi = cmp.median_os_range
print(f'\nmedian OS ranges {lo:.0f}-{hi:.0f} wk on the interaction assumption alone')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for label, tr in cmp.trajectories.items():
    ax.plot(t, tr.os_curve, lw=1.7, label=f'{label} (mOS {tr.median_os:.0f})' if tr.median_os else label)
ax.axhline(0.5, ls=':', color='grey')
ax.set(title=f'Population OS by interaction model - divergence {cmp.os_divergence:.2f}',
       xlabel='weeks', ylabel='survival fraction', ylim=(0, 1.02))
ax.legend(fontsize=8)
plt.show()

## Guardrails: synergy is an assumption, not a finding

Onkos does not estimate the interaction parameter — distinguishing synergy from additivity needs a combination trial designed for it. A non-zero ψ carries a warning; the underlying TGI model's tier governs and cannot be raised; and an inactive partner reduces to monotherapy under every model (no manufactured interaction).

In [ ]:
# Non-zero psi is flagged as an assumption.
assert cmp.warnings and 'combination trial' in cmp.warnings[0]

# Tier cannot be raised by the interaction assumption.
assert cmp.tier == ds['resistance.claret_2009.tgi'].tier

# Single-agent degeneracy: an inactive partner -> monotherapy under every model -> no divergence.
mono = compare_interactions(ds, 'resistance.claret_2009.tgi', context=ctx,
                            effect_a=0.6, effect_b=0.0, psi=0.5, t=t)
print('combined effects with an inactive partner:', {k: round(v, 3) for k, v in mono.combined_effects.items()})
print('OS divergence (should be 0):', round(mono.os_divergence, 6))
assert np.isclose(mono.os_divergence, 0.0)

d = cmp.to_dict()
assert d['NOT_FOR_CLINICAL_USE'] is True and 'PROHIBITED' in d['onkos:clinicalUse']
print('guardrails OK')